# 04 · Online softmax, tiled FlashAttention (in plain PyTorch), roofline

**Needed for:** FlashAttention-2 (LLM track, step 3). The same "running max and running sum" state shows up in
sparse attention (notebook 11), split-KV decoding, and ring attention.

Standard attention writes the full $T \times T$ score matrix to GPU memory (HBM), reads it back for softmax,
writes the probabilities, and reads them again for the multiply with V. FlashAttention never materializes the
matrix: it streams K/V in blocks through fast on-chip memory (SRAM) and keeps only a running softmax state
per query row. Same result, far fewer memory reads and writes.

This notebook builds the algorithm in plain PyTorch so you can check every step. Notebook 05 moves to Triton.

In [ ]:
import math
import torch
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. Online softmax

Safe softmax needs three passes: find the max, sum the exponentials, normalize. The online version fuses the
first two. Keep a running max $m$ and running sum $\ell$; when a new element raises the max, rescale the old sum:

$$m' = \max(m, x_i), \qquad \ell' = \ell\, e^{m - m'} + e^{x_i - m'}$$

In [ ]:
def online_softmax(x):
    """1-D tensor. One pass for (max, sum), one pass to write the output."""
    m, l = float("-inf"), 0.0
    for xi in x.tolist():
        m_new = max(m, xi)
        l = l * math.exp(m - m_new) + math.exp(xi - m_new)
        m = m_new
    return torch.exp(x - m) / l


def test_online_softmax(fn):
    x = torch.randn(50, dtype=torch.float64) * 10
    torch.testing.assert_close(fn(x), torch.softmax(x, 0))
    big = torch.tensor([1000.0, 1001.0, 1002.0], dtype=torch.float64)  # naive exp overflows here
    torch.testing.assert_close(fn(big), torch.softmax(big, 0))


test_online_softmax(online_softmax)
print("online softmax ok")

## 2. Merging two partial states

The state $(m, \ell)$ from two chunks can be combined, and the merge is associative. That's what lets
FlashAttention process K/V blocks in any grouping, and lets split-KV decoding (FlashDecoding, LeanAttention)
spread one long sequence across many GPU blocks and reduce the results at the end.

In [ ]:
def partial_state(x):
    m = x.max()
    return m, torch.exp(x - m).sum()


def merge_states(a, b):
    (ma, la), (mb, lb) = a, b
    m = torch.maximum(ma, mb)
    return m, la * torch.exp(ma - m) + lb * torch.exp(mb - m)


x = torch.randn(100, dtype=torch.float64)
m, l = merge_states(partial_state(x[:30]), partial_state(x[30:]))
torch.testing.assert_close(m + torch.log(l), torch.logsumexp(x, 0))
print("merged state gives the exact logsumexp")

## 3. FlashAttention forward, tiled

For each block of queries, loop over blocks of keys. Alongside $(m, \ell)$, keep an output accumulator
`acc` that is rescaled by the same factor $\alpha = e^{m - m'}$ whenever the max grows:

```
for each K/V block:
    S     = Q_blk @ K_blkᵀ * scale
    m_new = max(m, rowmax(S))
    P     = exp(S - m_new)
    α     = exp(m - m_new)
    ℓ     = α·ℓ + rowsum(P)
    acc   = α·acc + P @ V_blk
out = acc / ℓ
```

Also save $\text{lse} = m + \log \ell$ per row. The backward pass uses it to recompute P block by block
instead of storing it. That's the other half of the memory saving.

With a causal mask, K/V blocks entirely in the future are skipped (about half the work for long sequences).

In [ ]:
def flash_attention_forward(q, k, v, causal=False, block_q=16, block_k=16):
    """q, k, v: (B, H, T, d). Returns (out, lse) with lse: (B, H, T)."""
    B, H, T, d = q.shape
    scale = d ** -0.5
    out = torch.empty_like(q)
    lse = torch.empty(B, H, T, device=q.device, dtype=torch.float32)
    for qs in range(0, T, block_q):
        qe = min(qs + block_q, T)
        qi = q[:, :, qs:qe].float() * scale
        m = torch.full((B, H, qe - qs), float("-inf"), device=q.device)
        l = torch.zeros(B, H, qe - qs, device=q.device)
        acc = torch.zeros(B, H, qe - qs, d, device=q.device)
        k_stop = qe if causal else T                      # skip K/V blocks entirely in the future
        for ks in range(0, k_stop, block_k):
            ke = min(ks + block_k, k_stop)
            s = qi @ k[:, :, ks:ke].float().transpose(-2, -1)      # (B, H, bq, bk)
            if causal:
                q_pos = torch.arange(qs, qe, device=q.device)[:, None]
                k_pos = torch.arange(ks, ke, device=q.device)[None, :]
                s = s.masked_fill(k_pos > q_pos, float("-inf"))
            m_new = torch.maximum(m, s.amax(-1))
            p = torch.exp(s - m_new[..., None])
            alpha = torch.exp(m - m_new)
            l = alpha * l + p.sum(-1)
            acc = alpha[..., None] * acc + p @ v[:, :, ks:ke].float()
            m = m_new
        out[:, :, qs:qe] = (acc / l[..., None]).to(q.dtype)
        lse[:, :, qs:qe] = m + torch.log(l)
    return out, lse


def test_flash(fn):
    for causal in (False, True):
        for T in (1, 16, 37):                      # 37: not a multiple of the block size
            q, k, v = (torch.randn(2, 3, T, 32, device=device) for _ in range(3))
            out, lse = fn(q, k, v, causal=causal)
            ref = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
            torch.testing.assert_close(out, ref, atol=1e-4, rtol=1e-4)
            s = q @ k.transpose(-2, -1) / math.sqrt(32)
            if causal:
                s = s.masked_fill(torch.ones(T, T, device=device).triu(1).bool(), float("-inf"))
            torch.testing.assert_close(lse, torch.logsumexp(s, -1), atol=1e-4, rtol=1e-4)


test_flash(flash_attention_forward)
print("tiled attention matches SDPA, causal and non-causal")

## 4. Why it is faster: memory traffic and the roofline

**Arithmetic intensity** = FLOPs / bytes moved to or from HBM. A kernel is *memory-bound* if its intensity is
below the GPU's ridge point (peak FLOP/s ÷ memory bandwidth), and *compute-bound* above it.

For one head with sequence length $T$ and head dim $d$ (bf16, 2 bytes per element):
- **Standard attention** reads/writes Q, K, V, O ($4Td$) plus the score and probability matrices several
  times (about $4T^2$ element reads and writes).
- **FlashAttention** reads Q, K, V once per pass and writes O: roughly $4Td$ elements, plus re-reading K/V
  once per query block.

Both do about $4T^2d$ FLOPs (two matmuls). Fill in your GPU's numbers from its spec sheet.

In [ ]:
PEAK_TFLOPS = 121    # dense bf16 tensor-core TFLOP/s. Roughly: L4 ~121, A100 ~312, H100 SXM ~990. Check your spec sheet.
BANDWIDTH_TBS = 0.3  # HBM/GDDR TB/s. Roughly: L4 ~0.3, A100-80GB ~2.0, H100 SXM ~3.35.


def attention_traffic(T, d, flash, block_q=128):
    """(FLOPs, bytes moved to/from HBM) for one head, bf16."""
    flops = 4 * T * T * d
    if flash:
        elems = 2 * T * d + (T // block_q) * 2 * T * d   # Q and O once; K and V once per query block
    else:
        elems = 4 * T * d + 4 * T * T                    # Q, K, V, O, plus scores and probs written and read
    return flops, elems * 2


ridge = PEAK_TFLOPS / BANDWIDTH_TBS
print(f"ridge point: {ridge:.0f} FLOPs/byte\n")
for T in (512, 4096, 32768):
    f_std, b_std = attention_traffic(T, 128, False)
    f_fl, b_fl = attention_traffic(T, 128, True)
    print(f"T={T:6d}  standard: {b_std / 1e6:8.1f} MB, {f_std / b_std:4.0f} FLOPs/B"
          f"   flash: {b_fl / 1e6:7.1f} MB, {f_fl / b_fl:4.0f} FLOPs/B")

Read the numbers this way:
- **Standard attention's** intensity is stuck near $d/2$ however long the sequence gets, and its bytes grow as
  $T^2$ because of the score matrix.
- **FlashAttention's** intensity is about the query block size: each K/V element loaded is reused by
  `block_q` queries.
- With $d = 128$ this simple count shows only about a 2× saving in HBM bytes. The bigger wins are elsewhere:
  1. **Memory capacity.** Standard attention stores the $T \times T$ probabilities for backward: at $T = 32768$
     that's 2 GB per head in bf16. FlashAttention stores only `lse`, which is $O(T)$.
  2. **Fusion.** Unfused PyTorch runs separate kernels for scale, mask, softmax and dropout, each re-reading the
     score matrix. The count above assumes the best case of two round trips.
  3. **Cache.** Most repeated K/V reads hit L2 instead of HBM, so real kernels reach 50–75% of peak FLOP/s.

## Things to be able to explain

- Why must `acc` be rescaled by $\alpha$ every block, not just at the end?
- What does the backward pass recompute, and why is recomputing cheaper than storing?
- What changed in FlashAttention-2? (Parallelize over the sequence too, not just batch × heads; fewer
  non-matmul FLOPs; better work split between warps.) FlashAttention-3 adds Hopper-specific async copies and FP8.
- During decoding (1 query token), why does FlashAttention help less, and what does split-KV decoding do?

## Drill

In [ ]:
def my_online_softmax(x):
    raise NotImplementedError


def my_flash_attention_forward(q, k, v, causal=False, block_q=16, block_k=16):
    """Return (out, lse). Handle T not divisible by the block sizes."""
    raise NotImplementedError


drill(test_online_softmax, my_online_softmax)
drill(test_flash, my_flash_attention_forward)